# Run the TTE map on MAAP DPS

Runs the TTE domain tiles that are not done yet, one job per tile, with the registered algorithm
`run_tte_map`, version `tte_map_v1.1`. Same version, run label and job tag as before, so the results go to
the same DPS STAC collection, mapped in `notebooks/TTE_review_map.ipynb`.

Run the cells in order. **Steps 1-3 only read. Only step 4 (cancel) and step 5 (submit) change anything,
and only with `confirm=True`.**

## 1. Setup

In [ ]:
import os, sys
import requests
import pandas as pd
from maap.maap import MAAP
maap = MAAP()
REPO = os.path.expanduser('~/code/tte')
sys.path.insert(0, f'{REPO}/lib')
import tte_tcc_sources as ts
import tte_dps as dps

ALGO_ID, ALGO_VERSION = 'run_tte_map', 'tte_map_v1.1'
COLLECTION = 'montesano__run_tte_map_13__tte_map_v1.1'
PROCESS_ID = dps.find_process(maap, ALGO_ID, ALGO_VERSION)

# tiles that touch the TTE domain (the others only make empty jobs)
px = pd.read_csv(f'{REPO}/dps/tte_tiles_domain_pixels.csv')
DOMAIN_TILES = px.loc[px.domain_pixels > 0, 'tile_num'].astype(int).tolist()
print(f'{ALGO_ID} {ALGO_VERSION}: process {PROCESS_ID} | {len(DOMAIN_TILES)} TTE domain tiles')

## 2. Choose the run and the worker

`RUN` sets the tree cover. `QUEUE` sets the worker. The Meta CHM reader uses 8 threads, so it can use
up to about 8 CPUs.

In [ ]:
maap.get_queues().json()

In [ ]:
COMMON = {
    'tile_num': '',
    'tcc_options': 'none',
    'tcc_paths': 'none',
    'out_name': 'none',
    'gee_emulation': 'true',
    'domain_path': 's3://maap-ops-workspace/shared/montesano/tte/static/tte_domain_v201902v2.vrt',
    'tiles_gpkg': 's3://maap-ops-workspace/shared/montesano/databank/tte_tiles.gpkg',
    'write_stac': 'true',
}
RUNS = {   # examples; add your own
    'hansen_2010_gee_version': dict(tcc_source='hansen', tcc_year='2010', tcc_options='version=GFC-2018-v1.6'),
    'hansen_latest':           dict(tcc_source='hansen', tcc_year='2026', tcc_options='version=GFC-2025-v1.13'),
    'meta_chm_v2_2m':          dict(tcc_source='meta_chm', tcc_year='2020', tcc_options='version=v2;ht_min=2'),
    'my_rasters':              dict(tcc_source='path', tcc_year='2020', tcc_paths='s3://<bucket>/<my_tcc>.vrt',
                                    tcc_options='resampling=average', out_name='mytcc2020'),
}
RUN = 'meta_chm_v2_2m'        # set to one of RUNS
assert RUN in RUNS, f'set RUN to one of {list(RUNS)}'
params = dict(COMMON, **RUNS[RUN])
nn = lambda v: None if v in ('', 'none', None) else v
LABEL = ts.tcc_label(params['tcc_source'], params['tcc_year'], nn(params['tcc_options']), nn(params['out_name']))
IDENTIFIER = f'tte_map_{LABEL}'
print(ts.tcc_description(params['tcc_source'], params['tcc_year'], nn(params['tcc_options'])))
print('label:', LABEL, '| job tag:', IDENTIFIER)

QUEUE = 'maap-dps-worker-8gb'      # maap.get_queues().json() lists the others
print('queue:', QUEUE)

## 3. What is left (read only)

Re-run this any time to see progress. The first time it asks for the status of every logged job; finished jobs are
not asked for again.

In [ ]:
plan = dps.check(maap, IDENTIFIER, LABEL, COLLECTION, DOMAIN_TILES)

## 4. Cancel what is still queued

So that those tiles can run on the new worker. Running jobs finish. Then re-run step 3 until nothing is queued.

In [ ]:
dps.cancel(maap, plan, confirm=False)
# to cancel: dps.cancel(maap, plan, submitted_before='2026-11-24T12:00', confirm=True)   # queued jobs sent before this time (UTC)

## 5. Submit the tiles to run

Only the "to run" tiles of step 3. For a trial, add `only=plan.to_run()[:3]`.

In [ ]:
# t = plan.to_run()
# dps.submit(maap, plan, PROCESS_ID, params, QUEUE, confirm=True, only=t[:1])
# dps.submit(maap, plan, PROCESS_ID, params, 'maap-dps-worker-8gb', confirm=True, only=t[1:2])

In [ ]:
dps.submit(maap, plan, PROCESS_ID, params, QUEUE, confirm=True)

---
# Other

**Tree cover sources** for `RUNS`:

In [ ]:
print(ts.list_sources())

**Once: publish the original c2010 map in the DPS STAC**

Your GEE map `projects/foreststructure/TTE_DATA_v201902v2`, downloaded as 4,809 one-degree COGs, is not in
the DPS STAC, so TiTiler can only map its 1 km overview (`lib/tte_overview.py`), not the 30 m tiles. MAAP's **DPS STAC Item Generator** (OGC process
`generate_stac_items`) lists a prefix and writes one STAC item per COG, pointing at the files where they
are (nothing is copied). Its items go to a collection named after MAAP's deployment
(`montesano__generate_stac_items_<n>__<version>`), shared with anything else you import, so the job tag
picks out these items: `notebooks/TTE_review_map.ipynb` finds them with `tr.find_reference()`.
Reading 4,809 file headers takes a while; submit once.

In [ ]:
REF_SOURCE = 's3://maap-ops-workspace/montesano/dps_output/run_download_gee_tiles/download_gee_tiles_v1/tte_montesano_2010_30m/'
REF_TAG = 'tte_data_v201902v2_reference'          # the review notebook looks for this job tag

gen = dps._json(maap.list_algorithms(algorithm_name='generate_stac_items', algorithm_version='0.3.2'))
gen = gen.get('processes', gen) if isinstance(gen, dict) else gen
GEN_PROCESS_ID = gen[0].get('processID') if gen else None
print('DPS STAC Item Generator processID:', GEN_PROCESS_ID)    # 134 (v0.3.2) when this was written

# r = maap.submit_job(GEN_PROCESS_ID, inputs={'source': REF_SOURCE, 'region': 'us-west-2', 'include_extensions': '.tif',
#                                             'exclude_extensions': '', 'config_json': ''},
#                     queue='maap-dps-worker-8gb', tag=REF_TAG)
# ref_job = dps._json(r); ref_job

In [ ]:
# dps._json(maap.get_job_status(ref_job.get('jobID') or ref_job.get('id')))

## What each STAC item records

See `lib/tte_stac.py`.

| Group | Fields |
|---|---|
| Identity | id `tte_<tcc label>_<tile_num>_<parameter signature>`; footprint = the 1-degree tile; datetime = `tcc_year` (Jan 1) |
| How it was made | `tte:tcc_source`, `tte:tcc_label`, `tte:tcc_year`, `tte:tcc_options`, `tte:tcc_paths`, `tte:tcc_description`, `tte:water`, `tte:domain`, `tte:gee_pyramid_emulation`, `tte:zone_thresholds_pct`, `tte:code_version` (git tag), `tte:method`; DPS adds `maap-dps:*` (user, algorithm, job tag) and `created` |
| What is in the file | asset `tte` (COG), band names and descriptions, `raster:bands` (float32, nodata NaN, 30 m), `classification:classes` on band 4 (values 1-11, names, colors), `proj:*` (polar LAEA grid, WKT2) |
| Per-tile summary | `tte:domain_km2`, `tte:classified_km2`, `tte:classified_share_of_domain`, `tte:mean_tcc_domain`, `tte:mean_abruptness`, `tte:class_km2`, `tte:group_share` |
| Citation | `sci:doi`, `sci:citation` (Montesano et al. 2020); collection license CC-BY-4.0 |

Re-running a tile with the same parameters replaces its item; other parameters (tree cover source,
year, versions, options) give a new item in the same collection.